# Model training and evaluation

Create a stratified holdout split, fit preprocessing on training data only, compare baseline classifiers, tune selected models with cross-validation, evaluate on the holdout set, and save the selected pipeline and metrics.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "student_mental_health.csv"
student_df = pd.read_csv(data_path)
features = student_df.drop(columns=["Student_ID", "Mental_Health_Risk"])
target = student_df["Mental_Health_Risk"]

features_train, features_test, target_train, target_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

print("Train shape:", features_train.shape)
print("Test shape:", features_test.shape)


In [ ]:
ordinal_columns = ["Physical_Activity", "Diet_Quality"]
numeric_columns = features_train.select_dtypes(include="number").columns.tolist()
categorical_columns = [
    column
    for column in features_train.select_dtypes(exclude="number").columns
    if column not in ordinal_columns
]

print(f"Variables num?riques ({len(numeric_columns)}):", numeric_columns)
print(f"Variables cat?gorielles ({len(categorical_columns)}):", categorical_columns)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

ordinal_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                categories=[["Low", "Moderate", "High"], ["Poor", "Average", "Good"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
        ),
    ]
)

label_encoder = LabelEncoder()
target_train_encoded = label_encoder.fit_transform(target_train)
target_test_encoded = label_encoder.transform(target_test)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("ordinal", ordinal_pipeline, ordinal_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ]
)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=1,
    ),
}


In [ ]:
baseline_results = []
for name, estimator in baseline_models.items():
    pipeline = Pipeline(
        [("preprocessor", preprocessor), ("model", estimator)]
    )
    scores = cross_val_score(
        pipeline,
        features_train,
        target_train_encoded,
        cv=5,
        scoring="f1_weighted",
        n_jobs=2,
    )
    baseline_results.append(
        {
            "Model": name,
            "CV F1 Mean": scores.mean(),
            "CV F1 Std": scores.std(),
        }
    )

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values("CV F1 Mean", ascending=False)
    .reset_index(drop=True)
)


In [ ]:
display(baseline_results_df.round(4))

plt.figure(figsize=(8, 4))
plt.bar(
    baseline_results_df["Model"],
    baseline_results_df["CV F1 Mean"],
    yerr=baseline_results_df["CV F1 Std"],
    capsize=4,
)
plt.ylabel("5-fold weighted F1")
plt.title("Baseline model comparison")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV

candidate_models = {
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

parameter_grids = {
    "Random Forest": {
        "model__n_estimators": [150, 250],
        "model__max_depth": [10, 15, 20],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__class_weight": ["balanced", "balanced_subsample"],
    },
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__solver": ["lbfgs", "saga"],
        "model__class_weight": ["balanced"],
    },
}

optimized_models = {}
tuning_results = []
for name, estimator in candidate_models.items():
    pipeline = Pipeline(
        [("preprocessor", preprocessor), ("model", estimator)]
    )
    search = GridSearchCV(
        pipeline,
        parameter_grids[name],
        cv=5,
        scoring="f1_weighted",
        n_jobs=2,
    )
    search.fit(features_train, target_train_encoded)
    optimized_models[name] = search.best_estimator_
    tuning_results.append(
        {
            "Model": name,
            "Best CV F1-Score": search.best_score_,
            "Best Params": search.best_params_,
        }
    )

tuning_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("Best CV F1-Score", ascending=False)
    .reset_index(drop=True)
)
display(tuning_results_df)


In [ ]:
best_model_name = tuning_results_df.loc[0, "Model"]
best_cv_f1 = tuning_results_df.loc[0, "Best CV F1-Score"]
best_model = optimized_models[best_model_name]
target_predicted_best = best_model.predict(features_test)

selected_test_metrics = {
    "accuracy": accuracy_score(target_test_encoded, target_predicted_best),
    "precision": precision_score(
        target_test_encoded,
        target_predicted_best,
        average="weighted",
        zero_division=0,
    ),
    "recall": recall_score(
        target_test_encoded,
        target_predicted_best,
        average="weighted",
        zero_division=0,
    ),
    "f1_score": f1_score(
        target_test_encoded,
        target_predicted_best,
        average="weighted",
        zero_division=0,
    ),
}

final_results_df = pd.DataFrame(
    [
        {
            "Model selected by CV": best_model_name,
            "CV F1-Score": best_cv_f1,
            "Test Accuracy": selected_test_metrics["accuracy"],
            "Test Precision": selected_test_metrics["precision"],
            "Test Recall": selected_test_metrics["recall"],
            "Test F1-Score": selected_test_metrics["f1_score"],
        }
    ]
)
display(final_results_df.round(4))
print(f"Mod?le s?lectionn? par validation crois?e : {best_model_name}")


In [ ]:
print(
    classification_report(
        target_test_encoded,
        target_predicted_best,
        target_names=label_encoder.classes_,
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_estimator(
    best_model,
    features_test,
    target_test_encoded,
    display_labels=label_encoder.classes_,
    cmap="Blues",
)
plt.title(f"Matrice de confusion - {best_model_name}")
plt.show()


In [ ]:
errors = features_test.copy()
errors["Actual"] = label_encoder.inverse_transform(target_test_encoded)
errors["Predicted"] = label_encoder.inverse_transform(target_predicted_best)
errors = errors[errors["Actual"] != errors["Predicted"]]

print("Nombre d'erreurs :", len(errors), "sur", len(features_test))
display(errors[["Actual", "Predicted"]].value_counts())
display(errors.head(5))


In [ ]:
import json
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, models_dir / "mental_health_pipeline.joblib")
joblib.dump(label_encoder, models_dir / "label_encoder.joblib")

class_labels = list(range(len(label_encoder.classes_)))
metrics = {
    "model_name": best_model_name,
    "accuracy": round(float(selected_test_metrics["accuracy"]), 4),
    "precision": round(float(selected_test_metrics["precision"]), 4),
    "recall": round(float(selected_test_metrics["recall"]), 4),
    "f1_score": round(float(selected_test_metrics["f1_score"]), 4),
    "classes": label_encoder.classes_.tolist(),
    "confusion_matrix": confusion_matrix(
        target_test_encoded,
        target_predicted_best,
        labels=class_labels,
    ).tolist(),
}

with (models_dir / "evaluation_metrics.json").open("w", encoding="utf-8") as metrics_file:
    json.dump(metrics, metrics_file, indent=4)

print("Mod?le, encodeur et m?triques enregistr?s.")
